Ollama-3B Quiz generation test.

In [ ]:
import os
# Force everything to ignore the AMD iGPU and use the RTX 4050!!!
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

triton_cache = os.path.expanduser("~/.triton/cache")
os.makedirs(triton_cache, exist_ok=True)
os.environ["TRITON_CACHE_DIR"] = triton_cache
os.environ["TRITON_ALWAYS_COMPILE"] = "0"
os.environ["TORCH_COMPILE_DISABLE"] = "1"

os.environ["UNSLOTH_LLAMA_CPP_BACKEND"] = "cuda"
os.environ["XFORMERS_FORCE_DISABLE_TRITON"] = "1"
os.environ["UNSLOTH_DISABLE_TRITON"] = "1"
os.environ["TRITON_DISABLE"] = "1"

import torch
torch.cuda.empty_cache()

from datasets import load_dataset
from unsloth import FastLanguageModel

dataset = load_dataset("tuandunghcmut/coding-mcq-reasoning")
max_seq_length = 2048 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit", 
    max_seq_length = max_seq_length,
    dtype = None,                                         
    load_in_4bit = True
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,  
    loftq_config = None, 
)

def format_teacher_prompts(examples):
    questions = examples["question"]
    choices = examples["choices"] 
    understandings = examples["teacher_understanding"]
    analyses = examples["teacher_analysis"]
    reasonings = examples["teacher_reasoning"]
    conclusions = examples["teacher_conclusion"]
    answers = examples["teacher_answer"]
    texts = []
    for q, ch, und, ana, rea, con, ans in zip(questions, choices, understandings, analyses, reasonings, conclusions, answers):
        text = (
            f"### Question:\n{q}\n\n"
            f"### Choices:\n{ch}\n\n"
            f"### Teacher Thought Process:\n"
            f"- **Understanding**: {und}\n"
            f"- **Analysis**: {ana}\n"
            f"- **Reasoning**: {rea}\n"
            f"- **Conclusion**: {con}\n\n"
            f"### Final Answer:\n{ans}"
        )
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(format_teacher_prompts, batched = True)
print("Data formatted successfully!")

from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"], 
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    
    packing = False,
    args = TrainingArguments(
        torch_compile=False,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 120, 
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

print("Starting training...")
trainer.train()

model.save_pretrained_gguf("my_mcq_model_gguf", tokenizer, quantization_method = "q4_k_m")
print("Ollama-ready GGUF and Modelfile exported successfully!")



c:\Users\aarya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0906 10:07:47.130000 22192 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0906 10:07:47.290000 22192 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


c:\Users\aarya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\unsloth\import_fixes.py:2124: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)


...

Checking cache directory for required files...


Unsloth: Copying 2 files from cache to `my_mcq_model_gguf`: 100%|██████████| 2/2 [00:10<00:00,  5.29s/it]


Successfully copied all 2 files from cache to `my_mcq_model_gguf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:23<00:00, 11.72s/it]


Unsloth: Merge process complete. Saved to `c:\PythonStuff\Python_Sem_V\PersonalOllamaExperiment\my_mcq_model_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10798-mix-659e406 (app-b10798-mix-659e406-windows-x64-cpu.zip) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['my_mcq_model_gguf_gguf\\Llama-3.2-3B-Instruct.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model fi

Model Test.

In [3]:
import ollama

MODEL = "ollama-quiz-gen"
PROMPT = "Write a quiz for functions in python. Generate only a single question."

print(f"Sending prompt to {MODEL}...")
response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": PROMPT}]
)

print("\nAnswer:")
print(response["message"]["content"])


Sending prompt to ollama-quiz-gen...

Answer:
**Functions in Python Quiz**

**Question:** Which of the following is a correct example of using the `map()` function in Python?

A) `numbers = [1, 2, 3, 4, 5]; double_numbers = map(lambda x: x * 2, numbers)`
B) `numbers = [1, 2, 3, 4, 5]; double_numbers = map(x: x * 2, numbers)`
C) `numbers = [1, 2, 3, 4, 5]; double_numbers = map(lambda x: x * 2, numbers)`

**Teacher Thought Process**
---------------------------

### Understanding
The question tests the ability to apply the `map()` function in Python, which applies a given function to each item of an iterable (like a list or tuple) and returns an iterator.

### Analysis
- A: Correct. The `map()` function applies a given function (defined by the `lambda` expression) to each element of the list `numbers`, effectively doubling each number.
- B: Incorrect. The syntax `x: x * 2` is not valid Python syntax. It should be `lambda x: x * 2`.
- C: Correct. Similar to A, the `map()` function applies 